# 第 1 章 Notebook：Tensor、Autograd 与训练循环

这个 notebook 对应 `lessons/01_pytorch_training_intuition.md`。

核心追问：模型为什么能从错误里变好？

## 1. Tensor 先看 shape

在 LLM 里，很多 bug 都来自 shape 想错。先养成一个习惯：每一步都知道张量长什么样。

In [ ]:
import torch

x = torch.randn(4, 2)
print(x)
print('x.shape =', x.shape)

## 2. Autograd：让 loss 反过来告诉参数怎么改

`loss.backward()` 会计算梯度，但不会更新参数。真正改参数的是 optimizer。

In [ ]:
w = torch.tensor([2.0], requires_grad=True)
target = torch.tensor([10.0])

prediction = w * 3
loss = (prediction - target).pow(2).mean()
loss.backward()

print('prediction =', prediction.item())
print('loss =', loss.item())
print('w.grad =', w.grad.item())

## 3. 运行本章的 MLP 示例

这个示例不是为了追求模型效果，而是为了看清楚训练循环。

In [ ]:
from torch import nn
from torch.utils.data import DataLoader

from src.training.simple_mlp import SimpleMLP, ToyClassificationDataset, evaluate, train_one_epoch

dataset = ToyClassificationDataset(num_samples=256, seed=0)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
model = SimpleMLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(1, 6):
    train_metrics = train_one_epoch(model, dataloader, optimizer, loss_fn)
    eval_metrics = evaluate(model, dataloader, loss_fn)
    print(epoch, train_metrics, eval_metrics)

## 4. 自查问题

- `loss.backward()` 和 `optimizer.step()` 的分工是什么？
- 为什么 `logits.shape` 是 `[batch_size, num_classes]`？
- 如果忘记 `optimizer.zero_grad()` 会发生什么？